# Complete Demo: Multi-Batch Pipeline with Nessie & Iceberg

**Interview Demo Scenario**

This demo simulates a production data pipeline that receives files throughout the day.

## Demo Plan

| Step | Action | What we demonstrate |
|-------|--------|-------------------|
| 1 | **Batch 1** (morning) | Initial ingestion, table creation |
| 2 | **Batch 2** (noon) | Append, incremental update |
| 3 | **Batch 3** (afternoon) | Append, data evolution |
| 4 | **Time Travel** | Query data at different points in time |
| 5 | **Rollback** | Revert to a previous state |
| 6 | **Branching** | Isolated development on dev branch |

**Prerequisites**:
- Nessie started: `docker run -p 19120:19120 ghcr.io/projectnessie/nessie:latest`
- Data uploaded to S3: `python scripts/upload_to_s3.py`

---
## INITIALIZATION

In [ ]:
# Configuration of the path to the project
import sys
import os

project_root = os.path.abspath("..")
src_path = os.path.join(project_root, "src")
if src_path not in sys.path:
    sys.path.append(src_path)

# Import of Spark modules and configuration
from lakehouse.spark_session import get_spark
from lakehouse.settings import AWS_BUCKET, NESSIE_URI

# Creation of the Spark session
spark = get_spark("demo-complete")

print("=" * 60)
print("LAKEHOUSE DEMO - INITIALIZATION")
print("=" * 60)

In [ ]:
# Configuration of the Nessie main branch
spark.conf.set("spark.sql.catalog.nessie.ref", "main")

# Creation of namespaces (medallion architecture layers)
spark.sql("CREATE NAMESPACE IF NOT EXISTS nessie.bronze")
spark.sql("CREATE NAMESPACE IF NOT EXISTS nessie.silver")
spark.sql("CREATE NAMESPACE IF NOT EXISTS nessie.gold")

print("Active branch: main")
print("Nessie namespaces: bronze, silver, gold")
print(f"S3 Bucket: {AWS_BUCKET}")

---
## BATCH 1 - FIRST INGESTION (MORNING)

**Scenario**: It's morning, the first sales file arrives.

File: `sales_batch_1.csv` (3,364 records)

In [ ]:
# Import of PySpark functions
from pyspark.sql.functions import (
    current_timestamp, current_date, lit, col,
    sum as spark_sum, round as spark_round, count
)

# Cleanup: drop tables if they exist (for a clean demo)
spark.sql("DROP TABLE IF EXISTS nessie.bronze.sales")
spark.sql("DROP TABLE IF EXISTS nessie.silver.sales")
spark.sql("DROP TABLE IF EXISTS nessie.gold.sales_by_category_region")

print("Existing tables dropped")

In [ ]:
# Reading the first batch from S3
batch1_path = f"s3a://{AWS_BUCKET}/raw/batches/sales_batch_001.csv"

batch1_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("quote", '"')
    .option("escape", '"')
    .option("multiLine", "true")
    .csv(batch1_path)
)

batch1_count = batch1_raw.count()
print(f"BATCH 1 read: {batch1_count:,} records")

In [ ]:
# Enrichment and creation of the Bronze table
# Adding metadata: ingestion_date, ingestion_ts, source_file, source_system, batch_id
batch1_bronze = (
    batch1_raw
    .withColumn("ingestion_date", current_date())        # Ingestion date
    .withColumn("ingestion_ts", current_timestamp())      # Ingestion timestamp
    .withColumn("source_file", lit("sales_batch_1.csv")) # Source file name
    .withColumn("source_system", lit("sales_system"))    # Source system
    .withColumn("batch_id", lit("batch_1"))              # Batch identifier
)

# Creation of the Bronze table with partitioning on ingestion_date
batch1_bronze.writeTo("nessie.bronze.sales").using("iceberg").partitionedBy(col("ingestion_date")).create()

print(f"BATCH 1 -> Bronze table created: {batch1_count:,} records")

In [ ]:
# Transformation to the Silver layer
# - Renaming columns to snake_case
# - Strong typing of numeric columns
# - Filtering null values
# - Removing duplicates on (order_id, product_id)
batch1_silver = (
    batch1_bronze
    .select(
        col("Order ID").alias("order_id"),
        col("Order Date").alias("order_date"),
        col("Ship Date").alias("ship_date"),
        col("Ship Mode").alias("ship_mode"),
        col("Customer ID").alias("customer_id"),
        col("Customer Name").alias("customer_name"),
        col("Segment").alias("segment"),
        col("Country").alias("country"),
        col("City").alias("city"),
        col("State").alias("state"),
        col("Postal Code").alias("postal_code"),
        col("Region").alias("region"),
        col("Product ID").alias("product_id"),
        col("Category").alias("category"),
        col("Sub-Category").alias("sub_category"),
        col("Product Name").alias("product_name"),
        col("Sales").cast("double").alias("sales"),
        col("Quantity").cast("int").alias("quantity"),
        col("Discount").cast("double").alias("discount"),
        col("Profit").cast("double").alias("profit"),
        col("ingestion_date"),
        col("ingestion_ts"),
        col("batch_id")
    )
    .filter(col("order_id").isNotNull())
    .filter(col("product_id").isNotNull())
    .filter(col("sales").isNotNull())
    .dropDuplicates(["order_id", "product_id"])
)

batch1_silver.writeTo("nessie.silver.sales").using("iceberg").create()

silver_count = batch1_silver.count()
print(f"BATCH 1 -> Silver table created: {silver_count:,} records")

In [ ]:
# Creation of the Gold table: aggregation by category and region
# This table contains aggregated KPIs for reporting
batch1_gold = (
    batch1_silver
    .groupBy("category", "region")
    .agg(
        spark_round(spark_sum("sales"), 2).alias("total_sales"),
        spark_round(spark_sum("profit"), 2).alias("total_profit"),
        spark_sum("quantity").alias("total_quantity"),
        count("order_id").alias("order_count")
    )
)

batch1_gold.writeTo("nessie.gold.sales_by_category_region").using("iceberg").create()

gold_count = batch1_gold.count()
print(f"BATCH 1 -> Gold table created: {gold_count:,} aggregates")

In [ ]:
# Display of results after Batch 1
print("\n" + "=" * 60)
print("RESULTS AFTER BATCH 1")
print("=" * 60)
print(f"Bronze: {spark.sql('SELECT COUNT(*) FROM nessie.bronze.sales').first()[0]:,} records")
print(f"Silver: {spark.sql('SELECT COUNT(*) FROM nessie.silver.sales').first()[0]:,} records")
print(f"Gold:   {spark.sql('SELECT COUNT(*) FROM nessie.gold.sales_by_category_region').first()[0]:,} aggregates")

print("\nTop 5 sales by category/region:")
spark.sql("""
    SELECT category, region, total_sales, order_count
    FROM nessie.gold.sales_by_category_region
    ORDER BY total_sales DESC
    LIMIT 5
""").show(truncate=False)

In [ ]:
# Saving snapshot IDs for Time Travel (next section)
# Each Iceberg operation creates a unique snapshot that allows reverting to that state
batch1_snapshot_bronze = spark.sql("""
    SELECT snapshot_id 
    FROM nessie.bronze.sales.snapshots 
    ORDER BY committed_at DESC 
    LIMIT 1
""").first()[0]

batch1_snapshot_silver = spark.sql("""
    SELECT snapshot_id 
    FROM nessie.silver.sales.snapshots 
    ORDER BY committed_at DESC 
    LIMIT 1
""").first()[0]

batch1_snapshot_gold = spark.sql("""
    SELECT snapshot_id 
    FROM nessie.gold.sales_by_category_region.snapshots 
    ORDER BY committed_at DESC 
    LIMIT 1
""").first()[0]

print("Snapshot ID Bronze (Batch 1):", batch1_snapshot_bronze)
print("Snapshot ID Silver (Batch 1):", batch1_snapshot_silver)
print("Snapshot ID Gold   (Batch 1):", batch1_snapshot_gold)
print("(Saved for Time Travel)")

---
## BATCH 2 - SECOND INGESTION (NOON)

**Scenario**: It's noon, a new sales file arrives.

File: `sales_batch_2.csv` (3,501 records)

**Key point**: New data is **appended** to existing data (APPEND mode).

In [ ]:
# Reading the second batch from S3
batch2_path = f"s3a://{AWS_BUCKET}/raw/batches/sales_batch_002.csv"

batch2_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("quote", '"')
    .option("escape", '"')
    .option("multiLine", "true")
    .csv(batch2_path)
)

batch2_count = batch2_raw.count()
print(f"BATCH 2 read: {batch2_count:,} records")

In [ ]:
# Enrichment and addition to the Bronze table (APPEND mode)
batch2_bronze = (
    batch2_raw
    .withColumn("ingestion_date", current_date())
    .withColumn("ingestion_ts", current_timestamp())
    .withColumn("source_file", lit("sales_batch_2.csv"))
    .withColumn("source_system", lit("sales_system"))
    .withColumn("batch_id", lit("batch_2"))
)

# APPEND mode: data is added to existing data
batch2_bronze.writeTo("nessie.bronze.sales").append()

new_bronze_count = spark.sql("SELECT COUNT(*) FROM nessie.bronze.sales").first()[0]
print(f"BATCH 2 -> Added to Bronze table (APPEND)")
print(f"New Bronze total: {new_bronze_count:,} (+{new_bronze_count - batch1_count})")

In [ ]:
# Transformation and addition to Silver
batch2_silver = (
    batch2_bronze
    .select(
        col("Order ID").alias("order_id"),
        col("Order Date").alias("order_date"),
        col("Ship Date").alias("ship_date"),
        col("Ship Mode").alias("ship_mode"),
        col("Customer ID").alias("customer_id"),
        col("Customer Name").alias("customer_name"),
        col("Segment").alias("segment"),
        col("Country").alias("country"),
        col("City").alias("city"),
        col("State").alias("state"),
        col("Postal Code").alias("postal_code"),
        col("Region").alias("region"),
        col("Product ID").alias("product_id"),
        col("Category").alias("category"),
        col("Sub-Category").alias("sub_category"),
        col("Product Name").alias("product_name"),
        col("Sales").cast("double").alias("sales"),
        col("Quantity").cast("int").alias("quantity"),
        col("Discount").cast("double").alias("discount"),
        col("Profit").cast("double").alias("profit"),
        col("ingestion_date"),
        col("ingestion_ts"),
        col("batch_id")
    )
    .filter(col("order_id").isNotNull())
    .filter(col("product_id").isNotNull())
    .filter(col("sales").isNotNull())
    .dropDuplicates(["order_id", "product_id"])
)

batch2_silver.writeTo("nessie.silver.sales").append()

new_silver_count = spark.sql("SELECT COUNT(*) FROM nessie.silver.sales").first()[0]
print(f"BATCH 2 -> Added to Silver table (APPEND)")
print(f"New Silver total: {new_silver_count:,} (+{new_silver_count - silver_count})")

In [ ]:
# Complete recalculation of the Gold table (replacement table)
# For aggregation tables, we recalculate everything from Silver
all_silver = spark.table("nessie.silver.sales")

new_gold = (
    all_silver
    .groupBy("category", "region")
    .agg(
        spark_round(spark_sum("sales"), 2).alias("total_sales"),
        spark_round(spark_sum("profit"), 2).alias("total_profit"),
        spark_sum("quantity").alias("total_quantity"),
        count("order_id").alias("order_count")
    )
)

# createOrReplace: replaces the table content
new_gold.writeTo("nessie.gold.sales_by_category_region").using("iceberg").createOrReplace()

print(f"Gold recalculated with all Silver data: {new_gold.count():,} aggregates")

In [ ]:
# Display of results after Batch 2
print("\n" + "=" * 60)
print("RESULTS AFTER BATCH 2")
print("=" * 60)
print(f"Bronze: {spark.sql('SELECT COUNT(*) FROM nessie.bronze.sales').first()[0]:,} records")
print(f"Silver: {spark.sql('SELECT COUNT(*) FROM nessie.silver.sales').first()[0]:,} records")
print(f"Gold:   {spark.sql('SELECT COUNT(*) FROM nessie.gold.sales_by_category_region').first()[0]:,} aggregates")

print("\nTop 5 sales by category/region:")
spark.sql("""
    SELECT category, region, total_sales, order_count
    FROM nessie.gold.sales_by_category_region
    ORDER BY total_sales DESC
    LIMIT 5
""").show(truncate=False)

In [ ]:
# Saving snapshot IDs for Batch 2
batch2_snapshot_bronze = spark.sql("""
    SELECT snapshot_id 
    FROM nessie.bronze.sales.snapshots 
    ORDER BY committed_at DESC 
    LIMIT 1
""").first()[0]

batch2_snapshot_silver = spark.sql("""
    SELECT snapshot_id 
    FROM nessie.silver.sales.snapshots 
    ORDER BY committed_at DESC 
    LIMIT 1
""").first()[0]

batch2_snapshot_gold = spark.sql("""
    SELECT snapshot_id 
    FROM nessie.gold.sales_by_category_region.snapshots 
    ORDER BY committed_at DESC 
    LIMIT 1
""").first()[0]

print("Snapshot ID Bronze (Batch 2):", batch2_snapshot_bronze)
print("Snapshot ID Silver (Batch 2):", batch2_snapshot_silver)
print("Snapshot ID Gold   (Batch 2):", batch2_snapshot_gold)

---
## BATCH 3 - THIRD INGESTION (AFTERNOON)

**Scenario**: It's afternoon, the last sales file arrives.

File: `sales_batch_3.csv` (3,500 records)

In [ ]:
# Reading the third batch from S3
batch3_path = f"s3a://{AWS_BUCKET}/raw/batches/sales_batch_003.csv"

batch3_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("quote", '"')
    .option("escape", '"')
    .option("multiLine", "true")
    .csv(batch3_path)
)

batch3_count = batch3_raw.count()
print(f"BATCH 3 read: {batch3_count:,} records")

In [ ]:
# Enrichment and addition to Bronze
batch3_bronze = (
    batch3_raw
    .withColumn("ingestion_date", current_date())
    .withColumn("ingestion_ts", current_timestamp())
    .withColumn("source_file", lit("sales_batch_3.csv"))
    .withColumn("source_system", lit("sales_system"))
    .withColumn("batch_id", lit("batch_3"))
)

batch3_bronze.writeTo("nessie.bronze.sales").append()

final_bronze_count = spark.sql("SELECT COUNT(*) FROM nessie.bronze.sales").first()[0]
print(f"BATCH 3 -> Added to Bronze table (APPEND)")
print(f"New Bronze total: {final_bronze_count:,} (+{final_bronze_count - batch1_count})")

In [ ]:
# Transformation and addition to Silver
batch3_silver = (
    batch3_bronze
    .select(
        col("Order ID").alias("order_id"),
        col("Order Date").alias("order_date"),
        col("Ship Date").alias("ship_date"),
        col("Ship Mode").alias("ship_mode"),
        col("Customer ID").alias("customer_id"),
        col("Customer Name").alias("customer_name"),
        col("Segment").alias("segment"),
        col("Country").alias("country"),
        col("City").alias("city"),
        col("State").alias("state"),
        col("Postal Code").alias("postal_code"),
        col("Region").alias("region"),
        col("Product ID").alias("product_id"),
        col("Category").alias("category"),
        col("Sub-Category").alias("sub_category"),
        col("Product Name").alias("product_name"),
        col("Sales").cast("double").alias("sales"),
        col("Quantity").cast("int").alias("quantity"),
        col("Discount").cast("double").alias("discount"),
        col("Profit").cast("double").alias("profit"),
        col("ingestion_date"),
        col("ingestion_ts"),
        col("batch_id")
    )
    .filter(col("order_id").isNotNull())
    .filter(col("product_id").isNotNull())
    .filter(col("sales").isNotNull())
    .dropDuplicates(["order_id", "product_id"])
)

batch3_silver.writeTo("nessie.silver.sales").append()

final_silver_count = spark.sql("SELECT COUNT(*) FROM nessie.silver.sales").first()[0]
print(f"BATCH 3 -> Added to Silver table (APPEND)")
print(f"New Silver total: {final_silver_count:,} (+{final_silver_count - batch1_count})")

In [ ]:
# Recalculation of Gold with all data
all_silver_final = spark.table("nessie.silver.sales")

final_gold = (
    all_silver_final
    .groupBy("category", "region")
    .agg(
        spark_round(spark_sum("sales"), 2).alias("total_sales"),
        spark_round(spark_sum("profit"), 2).alias("total_profit"),
        spark_sum("quantity").alias("total_quantity"),
        count("order_id").alias("order_count")
    )
)

final_gold.writeTo("nessie.gold.sales_by_category_region").using("iceberg").createOrReplace()

print(f"Final Gold recalculated: {final_gold.count():,} aggregates")

In [ ]:
# Display of final results
print("\n" + "=" * 60)
print("RESULTS AFTER BATCH 3 (FINAL)")
print("=" * 60)
print(f"Bronze: {final_bronze_count:,} records")
print(f"Silver: {final_silver_count:,} records")
print(f"Gold:   {final_gold.count():,} aggregates")

print("\nTop 5 sales by category/region:")
spark.sql("""
    SELECT category, region, total_sales, order_count
    FROM nessie.gold.sales_by_category_region
    ORDER BY total_sales DESC
    LIMIT 5
""").show(truncate=False)

---
## TIME TRAVEL - TRAVEL BACK IN TIME

**Key point**: With Iceberg, you can query data as it was at any point in the past.

In [ ]:
# Display of complete Bronze snapshots history
print("=" * 60)
print("BRONZE SNAPSHOTS HISTORY")
print("=" * 60 + "\n")

spark.sql("""
    SELECT 
        made_current_at,
        snapshot_id,
        parent_id
    FROM nessie.bronze.sales.history
    ORDER BY made_current_at
""").show(truncate=False)

In [ ]:
# Time Travel 1: Query data from Batch 1 only
print("\n" + "=" * 60)
print("TIME TRAVEL: DATA AFTER BATCH 1")
print("=" * 60 + "\n")

# Using VERSION AS OF to query a past state
count_after_batch1 = spark.sql(f"""
    SELECT COUNT(*) as cnt 
    FROM nessie.bronze.sales VERSION AS OF '{batch1_snapshot_bronze}'
""").first()[0]

print(f"Number of records after Batch 1: {count_after_batch1:,}")

spark.sql(f"""
    SELECT batch_id, COUNT(*) as count
    FROM nessie.bronze.sales VERSION AS OF '{batch1_snapshot_bronze}'
    GROUP BY batch_id
    ORDER BY batch_id
""").show()

In [ ]:
# Time Travel 2: Query data from Batch 2
print("\n" + "=" * 60)
print("TIME TRAVEL: DATA AFTER BATCH 2")
print("=" * 60 + "\n")

count_after_batch2 = spark.sql(f"""
    SELECT COUNT(*) as cnt 
    FROM nessie.bronze.sales VERSION AS OF '{batch2_snapshot_bronze}'
""").first()[0]

print(f"Number of records after Batch 2: {count_after_batch2:,}")

spark.sql(f"""
    SELECT batch_id, COUNT(*) as count
    FROM nessie.bronze.sales VERSION AS OF '{batch2_snapshot_bronze}'
    GROUP BY batch_id
    ORDER BY batch_id
""").show()

In [ ]:
# Time Travel 3: Current data (after Batch 3)
print("\n" + "=" * 60)
print("CURRENT DATA (AFTER BATCH 3)")
print("=" * 60 + "\n")

count_after_batch3 = spark.sql("""
    SELECT COUNT(*) as cnt
    FROM nessie.bronze.sales
""").first()[0]

print(f"Number of records after Batch 3: {count_after_batch3:,}")

spark.sql("""
    SELECT batch_id, COUNT(*) as count
    FROM nessie.bronze.sales
    GROUP BY batch_id
    ORDER BY batch_id
""").show()

print("Observation: Each batch creates a new snapshot!")
print("We can go back to any point in time.")

In [ ]:
# Comparison of Gold aggregates over time
print("\n" + "=" * 60)
print("TECHNOLOGY/EAST SALES EVOLUTION")
print("=" * 60 + "\n")

# Technology/East sales after Batch 1 (using the GOLD snapshot)
sales_batch1 = spark.sql(f"""
    SELECT total_sales 
    FROM nessie.gold.sales_by_category_region VERSION AS OF '{batch1_snapshot_gold}'
    WHERE category = 'Technology' AND region = 'East'
""").first()[0]

# Current Technology/East sales
sales_current = spark.sql("""
    SELECT total_sales 
    FROM nessie.gold.sales_by_category_region
    WHERE category = 'Technology' AND region = 'East'
""").first()[0]

print(f"Technology/East sales after Batch 1: {sales_batch1:,.2f}")
print(f"Current Technology/East sales:     {sales_current:,.2f}")
print(f"Increase:                          {sales_current - sales_batch1:,.2f}")

---
## ROLLBACK - REVERT TO A PREVIOUS STATE

**Key point**: With Iceberg, you can revert to a previous state in case of error or problem.

In [ ]:
# Current state before rollback (after Batch 3)
print("=" * 60)
print("CURRENT STATE (AFTER BATCH 3)")
print("=" * 60 + "\n")

bronze_count = spark.sql("SELECT COUNT(*) FROM nessie.bronze.sales").first()[0]
silver_count = spark.sql("SELECT COUNT(*) FROM nessie.silver.sales").first()[0]
gold_count = spark.sql("SELECT COUNT(*) FROM nessie.gold.sales_by_category_region").first()[0]

print(f"Bronze: {bronze_count:,} records")
print(f"Silver: {silver_count:,} records")
print(f"Gold:   {gold_count:,} aggregates")

current_snapshot = spark.sql("""
    SELECT snapshot_id 
    FROM nessie.bronze.sales.snapshots 
    ORDER BY committed_at DESC 
    LIMIT 1
""").first()[0]

print(f"Current snapshot: {current_snapshot}")

In [ ]:
# ROLLBACK to the state after Batch 2
print("\n" + "=" * 60)
print("ROLLBACK TO STATE AFTER BATCH 2")
print("=" * 60 + "\n")

print(f"Target snapshot: {batch2_snapshot_bronze}")

# Reading data at Batch 2 state
bronze_batch2 = spark.sql(f"""
    SELECT * FROM nessie.bronze.sales VERSION AS OF '{batch2_snapshot_bronze}'
""")

count_batch2 = bronze_batch2.count()
print(f"Data to restore: {count_batch2:,} records")

# Rollback: replacing table content with Batch 2 state
bronze_batch2.writeTo("nessie.bronze.sales").using("iceberg").createOrReplace()

print("\nBronze rollback completed!")

In [ ]:
# Verification of state after Bronze rollback
print("\n" + "=" * 60)
print("STATE AFTER BRONZE ROLLBACK")
print("=" * 60 + "\n")

bronze_after_rollback = spark.sql("SELECT COUNT(*) FROM nessie.bronze.sales").first()[0]
print(f"Bronze: {bronze_after_rollback:,} records")
print(f"(Before rollback: {bronze_count:,})")
print(f"Difference: {bronze_count - bronze_after_rollback:,} records deleted")

print("\nBatches present after rollback:")
spark.sql("""
    SELECT batch_id, COUNT(*) as count
    FROM nessie.bronze.sales
    GROUP BY batch_id
    ORDER BY batch_id
""").show()

In [ ]:
# Complete ROLLBACK: Silver and Gold tables
print("\n" + "=" * 60)
print("ROLLBACK OF SILVER AND GOLD TABLES")
print("=" * 60 + "\n")

# Rollback Silver
silver_batch2 = spark.sql(f"""
    SELECT * FROM nessie.silver.sales VERSION AS OF '{batch2_snapshot_silver}'
""")
silver_batch2.writeTo("nessie.silver.sales").using("iceberg").createOrReplace()
print("Silver rollback completed!")

# Rollback Gold
gold_batch2 = spark.sql(f"""
    SELECT * FROM nessie.gold.sales_by_category_region VERSION AS OF '{batch2_snapshot_gold}'
""")
gold_batch2.writeTo("nessie.gold.sales_by_category_region").using("iceberg").createOrReplace()

print("Gold rollback completed!")

In [ ]:
# Verification of state after Silver and Gold rollback
print("\n" + "=" * 60)
print("STATE AFTER SILVER AND GOLD ROLLBACK")
print("=" * 60 + "\n")

silver_after_rollback = spark.sql("SELECT COUNT(*) FROM nessie.silver.sales").first()[0]
gold_after_rollback = spark.sql("SELECT COUNT(*) FROM nessie.gold.sales_by_category_region").first()[0]

print(f"Silver: {silver_after_rollback:,} records")
print(f"(Before rollback: {silver_count:,})")
print(f"Difference: {silver_count - silver_after_rollback:,} records deleted\n")

print(f"Gold: {gold_after_rollback:,} aggregates")
print(f"(Before rollback: {gold_count:,})")

print("\nBatches present in Silver after rollback:")
spark.sql("""
    SELECT batch_id, COUNT(*) as count
    FROM nessie.silver.sales
    GROUP BY batch_id
    ORDER BY batch_id
""").show()

print("Preview of Gold data after rollback:")
spark.sql("""
    SELECT category, region, total_sales, order_count
    FROM nessie.gold.sales_by_category_region
    ORDER BY total_sales DESC
    LIMIT 5
""").show(truncate=False)

print("\nAll rollbacks completed and verified!")
print("Data has returned to the state after Batch 2.")

---
## NESSIE BRANCHING - GIT-LIKE VERSIONING FOR DATA LAKES

### Concept

**Nessie** brings Git-like versioning to data lake tables. Like Git for code, Nessie allows:

* creating **isolated branches** to experiment
* testing data transformations without impacting production
* **merging** validated changes into the main branch

### Use Case

A data engineer wants to test a new transformation or ingest a new data batch **without risking** production cycles. They create a working branch, run tests, then merge if everything is OK.

```
┌─────────┐         ┌─────────┐
│  main   │ ←------ │   dev   │
│ (prod)  │  merge  │ (test)  │
└─────────┘         └─────────┘
```

#### 1. List existing references (branches)

**Command**: `LIST REFERENCES IN nessie`

This command displays all branches present in the Nessie catalog. By default, only the `main` branch exists.

In [ ]:
# Cleanup of the dev branch if it exists (for a clean demo)
spark.sql("DROP BRANCH IF EXISTS dev IN nessie")

print("=" * 60)
print("1. LIST OF NESSIE REFERENCES (BRANCHES)")
print("=" * 60 + "\n")

spark.sql("LIST REFERENCES IN nessie").show(truncate=False)

print("Explanation:")
print("- Displays all branches in the Nessie catalog")
print("- By default, only 'main' exists (production branch)")

#### 2. View the active branch

**Command**: `SHOW REFERENCE IN nessie`

This command displays the branch we are currently on. All SQL operations will be performed on this branch.

In [ ]:
# Switch back to main branch
spark.sql("USE REFERENCE main IN nessie")

print("\n" + "=" * 60)
print("2. ACTIVE BRANCH")
print("=" * 60 + "\n")

spark.sql("SHOW REFERENCE IN nessie").show(truncate=False)

current_ref = spark.sql("SHOW REFERENCE IN nessie").first()[1]
print(f"We are on branch: {current_ref}")
print("All SQL operations will affect this branch until we switch to another branch.")

#### 3. Create a working branch

**Command**: `CREATE BRANCH dev IN nessie FROM main`

Creates a new branch named `dev` from `main`. This branch is initially identical to `main` (instant copy, no data copy).

In [ ]:
print("\n" + "=" * 60)
print("3. CREATION OF THE 'dev' BRANCH")
print("=" * 60 + "\n")

try:
    spark.sql("CREATE BRANCH dev IN nessie FROM main").show(truncate=False)
    print("Branch 'dev' created successfully!")
except Exception as e:
    if "already exists" in str(e).lower():
        print("\nThe 'dev' branch already exists (created previously)")
    else:
        print(f"\nInfo: {e}")

print("\nExplanation:")
print("- The 'dev' branch is a logical copy of 'main'")
print("- No physical data copy (instant)")
print("- Changes on 'dev' will not impact 'main'")

#### 4. Switch to the dev branch

**Command**: `USE REFERENCE dev IN nessie`

Changes the active branch to `dev`. Now, all operations (INSERT, UPDATE, CREATE TABLE) will be performed on this branch, not on `main`.

In [ ]:
print("\n" + "=" * 60)
print("4. SWITCH TO THE 'dev' BRANCH")
print("=" * 60 + "\n")

spark.sql("USE REFERENCE dev IN nessie")

# Verification that we are on dev
spark.sql("SHOW REFERENCE IN nessie").show(truncate=False)

print("Explanation:")
print("- We are now on the 'dev' branch")
print("- All future operations will affect 'dev'")
print("- The 'main' branch remains intact (production safe)")

#### 5. Make a modification on the dev branch

Now that we are on `dev`, let's add Batch 3 to test our pipeline. This modification will not impact `main`.

In [ ]:
print("\n" + "=" * 60)
print("5. MODIFICATION ON THE 'dev' BRANCH (ADDING BATCH 3)")
print("=" * 60 + "\n")

# Verification of current state on dev
count_dev_before = spark.sql("SELECT COUNT(*) FROM nessie.bronze.sales").first()[0]
print(f"Bronze data on 'dev' before: {count_dev_before:,} records")

# Reading and adding Batch 3 on the dev branch
batch3_path = f"s3a://{AWS_BUCKET}/raw/batches/sales_batch_003.csv"

batch3_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("quote", '"')
    .option("escape", '"')
    .option("multiLine", "true")
    .csv(batch3_path)
)

batch3_bronze = (
    batch3_raw
    .withColumn("ingestion_date", current_date())
    .withColumn("ingestion_ts", current_timestamp())
    .withColumn("source_file", lit("sales_batch_3.csv"))
    .withColumn("source_system", lit("sales_system"))
    .withColumn("batch_id", lit("batch_3"))
)

batch3_bronze.writeTo("nessie.bronze.sales").append()

count_dev_after = spark.sql("SELECT COUNT(*) FROM nessie.bronze.sales").first()[0]
print(f"Bronze data on 'dev' after:  {count_dev_after:,} records")
print(f"Difference: +{count_dev_after - count_dev_before:,} records")

print("\nThis modification is only on 'dev' - 'main' is not impacted!")

#### 6. Compare main vs dev

Let's verify that `main` has not been modified by comparing data from both branches.

In [ ]:
print("\n" + "=" * 60)
print("6. COMPARISON: main vs dev")
print("=" * 60 + "\n")

# Data on dev (where we are)
dev_count = spark.sql("SELECT COUNT(*) FROM nessie.bronze.sales").first()[0]

# Switch to main to compare
spark.sql("USE REFERENCE main IN nessie")
main_count = spark.sql("SELECT COUNT(*) FROM nessie.bronze.sales").first()[0]

print(f"BRANCH 'dev' (with Batch 3): {dev_count:,} records")
print(f"BRANCH 'main' (production):  {main_count:,} records")
print(f"\nDifference: {dev_count - main_count:,} more records on 'dev'")

# Display batches present on main
print("\nBatches present on 'main':")
spark.sql("""
    SELECT batch_id, COUNT(*) as count
    FROM nessie.bronze.sales
    GROUP BY batch_id
    ORDER BY batch_id
""").show()

print("The 'main' branch is intact! Changes on 'dev' have not impacted production.")

#### 7. Display commit history

**Command**: `SHOW LOG IN nessie`

Displays the commit history on the active branch, similar to `git log`. We see all operations performed with their metadata.

In [ ]:
print("\n" + "=" * 60)
print("7. COMMIT HISTORY ON 'main'")
print("=" * 60 + "\n")

spark.sql("SHOW LOG IN nessie").show(truncate=False)

print("Explanation:")
print("- Each operation (CREATE, INSERT, MERGE) creates a commit")
print("- Commits have an ID, author, timestamp, and message")
print("- Like Git, we can navigate the history!")

#### 8. Merge the dev branch into main

**Command**: `MERGE BRANCH dev INTO main IN nessie`

Once tests are validated on `dev`, we merge the branch into `main`. This applies all changes from `dev` to production.

In [ ]:
print("\n" + "=" * 60)
print("8. MERGE 'dev' INTO 'main'")
print("=" * 60 + "\n")

print("Merging the branch into 'main' to deploy to production.")

spark.sql("MERGE BRANCH dev INTO main IN nessie").show(truncate=False)

# Important: refresh metadata after a Nessie merge
spark.sql("REFRESH TABLE nessie.bronze.sales")

# Verification
main_count_after = spark.sql("SELECT COUNT(*) FROM nessie.bronze.sales").first()[0]
print(f"\n'main' after merge: {main_count_after:,} records")
print(f"(before merge: {main_count:,}, +{main_count_after - main_count:,} new)")

print("\nBatches present on 'main' after merge:")
spark.sql("""
    SELECT batch_id, COUNT(*) as count
    FROM nessie.bronze.sales
    GROUP BY batch_id
    ORDER BY batch_id
""").show()

print("Batch 3 is now present in production!")

---
## DEMO SUMMARY

### Demonstrated Features

| Feature | Description |
|---------------|-------------|
| **Multi-batch ingestion** | Progressive data addition with APPEND mode |
| **Time Travel** | Query data at any point in the past via snapshots |
| **Rollback** | Revert to a previous state (e.g., cancel a batch) |
| **Branching** | Create isolated branches to experiment without risk |
| **Merge** | Merge an experimental branch into main |

### Benefits of Nessie + Iceberg + AWS

* **Isolation**: Work on a dev branch without impacting production
* **Safety**: Cancel errors via rollback
* **Audit**: Complete history of all modifications
* **Flexibility**: Test transformations before deploying them

**End of demo!**